# Description

In this notebook, we will train the NLP model using news dataset

In [1]:
import os 
import json
import math
import numpy as np
import pandas as pd
from typing import Optional

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# 1. Read dataset

In [2]:
PATH_JSON_DATA = "/home/tnguyen10/Desktop/quantization/news_dataset/News_Category_Dataset_v3.json"

with open(PATH_JSON_DATA, 'r') as f:
    data = f.readlines()
    
data = [json.loads(line) for line in data]
print(f"Number of news articles: {len(data)}")

Number of news articles: 209527


In [3]:
# Only get short_description and category. Then convert to pandas 
data = [({'description': item['short_description'], 'category': item['category']}) for item in data]

df = pd.DataFrame(data)
df.head()

,description,category
0,Health experts said it is too early to predict...,U.S. NEWS
1,He was subdued by passengers and crew when he ...,U.S. NEWS
2,"""Until you have a dog you don't understand wha...",COMEDY
3,"""Accidentally put grown-up toothpaste on my to...",PARENTING
4,Amy Cooper accused investment firm Franklin Te...,U.S. NEWS


In [4]:
# Only select POLITICS, WELLNESS, ENTERTAINMENT, TRAVEL, FOOD & DRINK, SPORTS categories
df = df[df['category'].isin(['POLITICS', 'TRAVEL', 'FOOD & DRINK'])]
print(f"Shape of dataframe after filtering: {df.shape}")

df.head()

Shape of dataframe after filtering: (51842, 2)


,description,category
21,President issues vow as tensions with China rise.,POLITICS
24,An annual celebration took on a different feel...,POLITICS
30,"U.S. President Joe Biden, in London for the fu...",POLITICS
40,Republican outrage over the shoddy U.S. withdr...,POLITICS
44,The former White House chief of staff has turn...,POLITICS


In [5]:
# Balancing dataset
min_count = df['category'].value_counts().min()
balanced_df = pd.DataFrame()

for category in df['category'].unique():
    category_df = df[df['category'] == category].sample(min_count, random_state=42)
    balanced_df = pd.concat([balanced_df, category_df])
    
df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle the dataframe
print(f"Shape of dataframe after balancing: {df.shape}")
df['category'].value_counts()

Shape of dataframe after balancing: (19020, 2)


category
TRAVEL          6340
FOOD & DRINK    6340
POLITICS        6340
Name: count, dtype: int64

Load Glove embedding

In [6]:
glove_path = "news_dataset/glove.6B.100d.txt"
embedding_dim = 100

# Step 1: Load GloVe vectors into a dictionary
glove_dict = {}
with open(glove_path, 'r', encoding='utf8') as f:
    for line in f:
        parts = line.strip().split()
        word = parts[0]
        vector = torch.tensor([float(v) for v in parts[1:]], dtype=torch.float32)
        glove_dict[word] = vector

print(f"Loaded {len(glove_dict)} words from GloVe.")

Loaded 400000 words from GloVe.


# 2. Data preparation

In [7]:
X = df['description'].values
y = df['category'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Number of training samples: {len(X_train)}")
print(f"Number of testing samples: {len(X_test)}")

Number of training samples: 15216
Number of testing samples: 3804


In [8]:
# convert label to integer
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Classes:", le.classes_)
print(f"Number of classes: {len(le.classes_)}")

print(f"Shape of y_train_enc: {y_train_enc.shape}")
print(f"Shape of y_test_enc: {y_test_enc.shape}")

Classes: ['FOOD & DRINK' 'POLITICS' 'TRAVEL']
Number of classes: 3
Shape of y_train_enc: (15216,)
Shape of y_test_enc: (3804,)


# 3. Model architecture

In [9]:
def dummy_int8_matmul(A_int8:torch.Tensor, B_int:torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int8.dtype != torch.int8 or B_int.dtype != torch.int8:
        raise ValueError("Both A and B must be int8 tensors.")
    result_float = torch.matmul(A_int8.float(), B_int.float())
    return result_float.to(out_dtype)

# Custom linear layer with quantization 
class Custom_LinearLayer_Quantized(nn.Module):
    def __init__(self, in_features: int, out_features: int, num_bits: int = 8):
        super(Custom_LinearLayer_Quantized, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_bits = num_bits
        self.weight = nn.Parameter(torch.Tensor(in_features, out_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        
        # Calibration buffers
        self.register_buffer('x_min', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('x_max', torch.tensor(0.0, dtype=torch.float32))
        self.calibrating = True
        
        # Quantization parameters
        self.register_buffer('w_q', torch.zeros((in_features, out_features), dtype=torch.int8))
        self.register_buffer('w_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('x_scale', torch.tensor(1.0, dtype=torch.float32))
        self.frozen = False  # Flag to indicate if weights are frozen
        
    @torch.no_grad()
    def observe_activation(self, x):
        min_val = x.min()
        max_val = x.max()
        self.x_min = min(self.x_min, min_val)
        self.x_max = max(self.x_max, max_val)
        

    @torch.no_grad()
    def compute_quantize_parameter(self, mat:torch.Tensor):
        """
        Symmetric quantization to int8.
        * Parameters:
            mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
            
        * Returns:
            q_mat: quantized int8 matrix
            scale: scale factor (torch.float32)
        """
        max_val = torch.max(torch.abs(mat))
        
        qmin = -128
        qmax = 127
        scale = max_val / qmax
        
        q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
        
        scale = scale.clone().detach().to(torch.float32)
        return q_mat, scale
    
    @torch.no_grad()
    def start_quantization_weight(self):
        # Compute activation quantization parameters
        x_max = max(abs(self.x_min), abs(self.x_max))
        x_scale = x_max / 127.0
        self.x_scale.copy_(x_scale)
        
        # Quantize weights
        W_q, w_scale = self.compute_quantize_parameter(self.weight)
        self.w_q.copy_(W_q)
        self.w_scale.copy_(w_scale)
        self.frozen = True
        print(f"[INFO] DONE quantizing weights. Weight scale: {self.w_scale.item():.6f}, Activation scale: {self.x_scale.item():.6f}")


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        # Normal float operation
        if not self.frozen and not self.calibrating:
            return torch.matmul(x, self.weight)
    
        # During calibration phase
        if not self.frozen and self.calibrating:
            self.observe_activation(x)
            return torch.matmul(x, self.weight)
        
        # Quantized matmul
        if x.dtype != torch.int8:
            X_q = torch.clamp(torch.round(x / self.x_scale), -128, 127).to(torch.int8)        
        else:
            X_q = x
        
        A_q = dummy_int8_matmul(X_q, self.w_q, out_dtype=torch.int32)
        A_deq = A_q.to(torch.float32) * (self.x_scale * self.w_scale)
        return A_deq 

In [ ]:
class Custom_MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention implemented from scratch using only matmul

    Shapes use batch-first convention: (B, T, D).

    Args:
        d_model: model (embedding) dimension.
        num_heads: number of attention heads.
        dropout_p: dropout probability on attention weights.
        bias: whether to include bias terms for the projections.
        causal: if True, applies a causal mask (no peeking ahead).
    """

    def __init__(self, d_model: int, num_heads: int, dropout_p: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.dropout_p = dropout_p

        # Projection weights
        # Input: (B, T, D) -> (B, T, D) per projection
        self.W_q = nn.Parameter(torch.empty(d_model, d_model))
        self.W_k = nn.Parameter(torch.empty(d_model, d_model))
        self.W_v = nn.Parameter(torch.empty(d_model, d_model))

        # Output projection
        self.W_o = nn.Parameter(torch.empty(d_model, d_model))

        # Initialize parameters
        for W in (self.W_q, self.W_k, self.W_v, self.W_o):
            nn.init.xavier_uniform_(W)

        # Calibration buffers
        self.register_buffer('xq_min', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('xq_max', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('xk_min', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('xk_max', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('xv_min', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('xv_max', torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer('x_o_min', torch.tensor(0.0, dtype=torch.float32)) # output of scaled dot-product
        self.register_buffer('x_o_max', torch.tensor(0.0, dtype=torch.float32)) # output of scaled dot-product
        self.calibrating = True
        
        # Quantization parameters 
        self.register_buffer('Wq_q', torch.zeros((d_model, d_model), dtype=torch.int8))
        self.register_buffer('Wk_q', torch.zeros((d_model, d_model), dtype=torch.int8))
        self.register_buffer('Wv_q', torch.zeros((d_model, d_model), dtype=torch.int8))
        self.register_buffer('Wo_q', torch.zeros((d_model, d_model), dtype=torch.int8))
        self.register_buffer('Wq_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('Wk_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('Wv_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('Wo_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('xq_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('xk_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('xv_scale', torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer('xo_scale', torch.tensor(1.0, dtype=torch.float32))
        self.frozen = False  # Flag to indicate if weights are frozen
        

    @staticmethod
    def _split_heads(x: torch.Tensor, num_heads: int) -> torch.Tensor:
        """
        (B, T, D_model) -> (B, H, T, Dh)
        """
        B, T, D = x.shape
        Dh = D // num_heads
        x = x.view(B, T, num_heads, Dh)
        return x.permute(0, 2, 1, 3)

    @staticmethod
    def _merge_heads(x: torch.Tensor) -> torch.Tensor:
        """
        (B, H, T, Dh) -> (B, T, H*Dh)
        """
        B, H, T, Dh = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(B, T, H * Dh)
    
    @torch.no_grad()
    def observe_qkv_activation(self, x_q, x_k, x_v):
        min_val_q = x_q.min()
        max_val_q = x_q.max()
        self.xq_min = min(self.xq_min, min_val_q)
        self.xq_max = max(self.xq_max, max_val_q)
        
        min_val_k = x_k.min()
        max_val_k = x_k.max()
        self.xk_min = min(self.xk_min, min_val_k)
        self.xk_max = max(self.xk_max, max_val_k)
        
        min_val_v = x_v.min()
        max_val_v = x_v.max()
        self.xv_min = min(self.xv_min, min_val_v)
        self.xv_max = max(self.xv_max, max_val_v)
        
    @torch.no_grad()
    def observe_output_activation(self, x_o):
        min_val_o = x_o.min()
        max_val_o = x_o.max()
        self.x_o_min = min(self.x_o_min, min_val_o)
        self.x_o_max = max(self.x_o_max, max_val_o)
        
    @torch.no_grad()
    def compute_quantize_parameter(self, mat:torch.Tensor):
        """
        Symmetric quantization to int8.
        * Parameters:
            mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
            
        * Returns:
            q_mat: quantized int8 matrix
            scale: scale factor (torch.float32)
        """
        max_val = torch.max(torch.abs(mat))
        
        qmin = -128
        qmax = 127
        scale = max_val / qmax
        
        q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
        
        scale = scale.clone().detach().to(torch.float32)
        return q_mat, scale
    
    @torch.no_grad()
    def start_quantization_weight(self):
        
        # Compute activation quantization parameters
        xq_max = max(abs(self.xq_min), abs(self.xq_max))
        xq_scale = xq_max / 127.0
        self.xq_scale.copy_(xq_scale)
        
        xk_max = max(abs(self.xk_min), abs(self.xk_max))
        xk_scale = xk_max / 127.0
        self.xk_scale.copy_(xk_scale)
        
        xv_max = max(abs(self.xv_min), abs(self.xv_max))
        xv_scale = xv_max / 127.0
        self.xv_scale.copy_(xv_scale)
        
        xo_max = max(abs(self.x_o_min), abs(self.x_o_max))
        xo_scale = xo_max / 127.0
        self.xo_scale.copy_(xo_scale)
        
        # Quantize weights
        Wq_q, Wq_scale = self.compute_quantize_parameter(self.W_q)
        self.Wq_q.copy_(Wq_q)
        self.Wq_scale.copy_(Wq_scale)
        
        Wk_q, Wk_scale = self.compute_quantize_parameter(self.W_k)
        self.Wk_q.copy_(Wk_q)
        self.Wk_scale.copy_(Wk_scale)
        
        Wv_q, Wv_scale = self.compute_quantize_parameter(self.W_v)
        self.Wv_q.copy_(Wv_q)
        self.Wv_scale.copy_(Wv_scale)
        
        Wo_q, Wo_scale = self.compute_quantize_parameter(self.W_o)
        self.Wo_q.copy_(Wo_q)
        self.Wo_scale.copy_(Wo_scale)
        
        self.frozen = True
        print(f"[INFO] DONE quantizing Wq_q, Wk_q, Wv_q, Wo_q.")
        
        # Delete float weights to save memory
        del self.W_q
        del self.W_k
        del self.W_v
        del self.W_o
        print(f"[INFO] Deleted float weights to save memory.")

    def forward(self, x_q: torch.Tensor, x_k: Optional[torch.Tensor] = None, x_v: Optional[torch.Tensor] = None):
        """
        Args:
            x_q: query input (B, T_q, D)
            x_kv: key/value input (B, T_k, D). If None, uses x_q (self-attention).
            key_padding_mask: boolean mask (B, T_kv), True for positions to mask (padding).  # TODO: implement
        Returns:
            y: output (B, T_q, D)
            attn_probs_avg (optional): (B, T_q, T_kv)
        """
        if x_k is None:
            x_k = x_q
        if x_v is None:
            x_v = x_q

        B, T_q, D = x_q.shape
        B2, T_k, D2 = x_k.shape
        assert D == self.d_model and D2 == self.d_model and B == B2

        # Projections via matmul
        if not self.frozen and not self.calibrating: # normal float operation
            Q = torch.matmul(x_q, self.W_q)  # (B, T_q, D)
            K = torch.matmul(x_k, self.W_k)  # (B, T_k, D)
            V = torch.matmul(x_v, self.W_v)  # (B, T_k, D)
        elif not self.frozen and self.calibrating: # calibration phase (float operation + observe activations)
            Q = torch.matmul(x_q, self.W_q)  # (B, T_q, D)
            K = torch.matmul(x_k, self.W_k)  # (B, T_k, D)
            V = torch.matmul(x_v, self.W_v)  # (B, T_k
            self.observe_qkv_activation(x_q, x_k, x_v) 
        else: # quantized matmul
            if x_q.dtype != torch.int8:
                xq_q = torch.clamp(torch.round(x_q / self.xq_scale), -128, 127).to(torch.int8)        
            else:
                xq_q = x_q
            
            if x_k.dtype != torch.int8:
                xk_q = torch.clamp(torch.round(x_k / self.xk_scale), -128, 127).to(torch.int8)        
            else:
                xk_q = x_k
            
            if x_v.dtype != torch.int8:
                xv_q = torch.clamp(torch.round(x_v / self.xv_scale), -128, 127).to(torch.int8)        
            else:
                xv_q = x_v
                
            Q_int32 = dummy_int8_matmul(xq_q, self.Wq_q, out_dtype=torch.int32)
            Q = Q_int32.to(torch.float32) * (self.xq_scale * self.Wq_scale)
            K_int32 = dummy_int8_matmul(xk_q, self.Wk_q, out_dtype=torch.int32)
            K = K_int32.to(torch.float32) * (self.xk_scale * self.Wk_scale)
            V_int32 = dummy_int8_matmul(xv_q, self.Wv_q, out_dtype=torch.int32)
            V = V_int32.to(torch.float32) * (self.xv_scale * self.Wv_scale)

        # Split into heads
        Q = self._split_heads(Q, self.num_heads)  # (B, H, T_q, Dh)
        K = self._split_heads(K, self.num_heads)  # (B, H, T_kv, Dh)
        V = self._split_heads(V, self.num_heads)  # (B, H, T_kv, Dh)

        # Scaled dot-product attention: (B, H, T_q, Dh) @ (B, H, Dh, T_kv) -> (B, H, T_q, T_kv)
        scale = 1.0 / math.sqrt(self.d_head)
        scores = torch.matmul(Q, K.transpose(-1, -2)) * scale

        attn_probs = F.softmax(scores, dim=-1)

        # (B, H, T_q, T_kv) @ (B, H, T_kv, Dh) -> (B, H, T_q, Dh)
        context = torch.matmul(attn_probs, V)

        # Merge heads and output projection
        context = self._merge_heads(context)  # (B, T_q, D)
        
        if not self.frozen and not self.calibrating:
            y = torch.matmul(context, self.W_o)  # (B, T_q, D)
        elif not self.frozen and self.calibrating:
            y = torch.matmul(context, self.W_o)  # (B, T_q, D)
            self.observe_output_activation(context)
        else:
            if context.dtype != torch.int8:
                context_q = torch.clamp(torch.round(context / self.xo_scale), -128, 127).to(torch.int8)        
            else:
                context_q = context
            y = dummy_int8_matmul(context_q, self.Wo_q, out_dtype=torch.int32)
            y = y.to(torch.float32) * (self.xo_scale * self.Wo_scale)
            
        return y

In [11]:
# model architecture with embedding layer and multi-head attention

class NewsClassifier(nn.Module):
    def __init__(self, hidden_dim, output_dim, embedding_matrix):
        super(NewsClassifier, self).__init__()
        
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True)
        embedding_dim = embedding_matrix.size(1)
        
        # self.attention_1 = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=4, batch_first=True)
        # self.attention_2 = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=4, batch_first=True)
        
        self.attention_1 = Custom_MultiHeadAttention(d_model=embedding_dim, num_heads=4)
        self.attention_2 = Custom_MultiHeadAttention(d_model=embedding_dim, num_heads=4)
        
        self.fc1 = Custom_LinearLayer_Quantized(embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = self.embedding(x)
        # x, _ = self.attention_1(x, x, x)
        # x, _ = self.attention_2(x, x, x)
        
        x = self.attention_1(x, x, x)  # custom attention does not return attn weights
        x = self.attention_2(x, x, x)  # custom attention does not return attn weights
        
        x = F.relu(self.fc1(x.mean(dim=1)))
        x = self.fc2(x)
        return x
    
    @torch.no_grad()
    def start_quantization(self):
        print("[INFO] === Start Post-Training quantization ==== ")
        self.attention_1.start_quantization_weight()
        print("\t Done quantizing attention layer 1.")
        
        self.attention_2.start_quantization_weight()
        print("\t Done quantizing attention layer 2.")
        
        self.fc1.start_quantization_weight()
        print("\t Done quantizing linear layers.")
        print("[INFO] === Quantization completed. ===")

# 4. Training process

In [12]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        input_ids = encoding['input_ids'].squeeze()
        
        return {
            'input_ids': input_ids,
            'label': torch.tensor(label, dtype=torch.long)
        }

In [13]:
# Simple tokenizer using space splitting
class SimpleTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.word2idx = {word: idx for idx, word in enumerate(vocab)}
        
    def __call__(self, text, max_length, padding, truncation, return_tensors):
        tokens = text.split()
        input_ids = [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]
        
        if truncation and len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
        
        if padding == 'max_length' and len(input_ids) < max_length:
            input_ids += [self.word2idx['<PAD>']] * (max_length - len(input_ids))
        
        return {'input_ids': torch.tensor([input_ids], dtype=torch.long)}

In [14]:
# Prepare tokenizer and dataset
vocab = ['<PAD>', '<UNK>'] + list(set(' '.join(df['description'].values).split()))
print(f"Vocabulary size: {len(vocab)}")

# Build embedding matrix
embedding_matrix = torch.zeros((len(vocab), embedding_dim))
for i, word in enumerate(vocab):
    if word in glove_dict:
        embedding_matrix[i] = glove_dict[word]
    else:
        embedding_matrix[i] = torch.randn(embedding_dim)  # Random init for unknown words
        
print(f"Embedding matrix shape: {embedding_matrix.shape}")

Vocabulary size: 51066
Embedding matrix shape: torch.Size([51066, 100])


In [15]:
tokenizer = SimpleTokenizer(vocab)
max_length = 128
batch_size = 128

train_dataset = NewsDataset(X_train, y_train_enc, tokenizer, max_length)
test_dataset = NewsDataset(X_test, y_test_enc, tokenizer, max_length)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Done preparing data loaders with batch size {batch_size}.")

Done preparing data loaders with batch size 128.


In [16]:
# Training hyper-parameters
hidden_dim = 64

model = NewsClassifier(hidden_dim=hidden_dim, output_dim=len(le.classes_),\
    embedding_matrix=embedding_matrix)

# print number of parameter per layer
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name} | Params: {param.numel()}")

# print(f"Number of model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Layer: attention_1.W_q | Params: 10000
Layer: attention_1.W_k | Params: 10000
Layer: attention_1.W_v | Params: 10000
Layer: attention_1.W_o | Params: 10000
Layer: attention_2.W_q | Params: 10000
Layer: attention_2.W_k | Params: 10000
Layer: attention_2.W_v | Params: 10000
Layer: attention_2.W_o | Params: 10000
Layer: fc1.weight | Params: 6400
Layer: fc2.weight | Params: 192
Layer: fc2.bias | Params: 3


In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

num_epochs = 5

In [18]:
# Training process
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids']
        labels = batch['label']
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids']
            labels = batch['label']
            
            outputs = model(input_ids)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch 1/5, Loss: 0.7346
Test Accuracy: 0.7708
Epoch 2/5, Loss: 0.5509
Test Accuracy: 0.7834
Epoch 3/5, Loss: 0.5218
Test Accuracy: 0.7905
Epoch 4/5, Loss: 0.4982
Test Accuracy: 0.7873
Epoch 5/5, Loss: 0.4841
Test Accuracy: 0.7881


In [19]:
# Evaluation on test set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids']
        labels = batch['label']
        
        outputs = model(input_ids)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
accuracy = correct / total
print(f"Final Test Accuracy: {accuracy:.4f}")

Final Test Accuracy: 0.7881


In [20]:
sample_text = """Pham Minh Chinh is the prime minister of the Vietnam goverment."""
# sample_text = """The new travel guidelines have been released for tourists. Vietnam is welcoming visitors back."""
# sample_text = """Pho is a popular dish, which is a noodle soup consisting of broth, rice noodles, herbs, and meat."""
sample_encoding = tokenizer(
    sample_text,
    max_length=max_length,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

sample_input_ids = sample_encoding['input_ids']
model.eval()
with torch.no_grad():
    sample_output = model(sample_input_ids)
    _, sample_predicted = torch.max(sample_output.data, 1)
    predicted_category = le.inverse_transform(sample_predicted.numpy())[0]
    
print(f"Sample Text: {sample_text}")
print(f"Predicted Category: {predicted_category}")

Sample Text: Pham Minh Chinh is the prime minister of the Vietnam goverment.
Predicted Category: POLITICS


In [21]:
# Save model except embedding layer
state_dict = model.state_dict()

# Remove the embedding weights
state_dict_no_embed = {k: v for k, v in state_dict.items() if not k.startswith("embedding.")}

# Save
output_path = os.path.join("models", "news_classifier_no_embedding.pth")
torch.save(state_dict_no_embed, output_path)

print(f"Model size without embedding layer. File size: {os.path.getsize(output_path) / (1024 * 1024):.2f} MB") 

Model size without embedding layer. File size: 0.43 MB


## 4.1. Post-Training quantization

In [22]:
model.eval()
model.start_quantization()

[INFO] === Start Post-Training quantization ==== 
[INFO] DONE quantizing Wq_q, Wk_q, Wv_q, Wo_q.
[INFO] Deleted float weights to save memory.
	 Done quantizing attention layer 1.
[INFO] DONE quantizing Wq_q, Wk_q, Wv_q, Wo_q.
[INFO] Deleted float weights to save memory.
	 Done quantizing attention layer 2.
[INFO] DONE quantizing weights. Weight scale: 0.001447, Activation scale: 0.068286
	 Done quantizing linear layers.
[INFO] === Quantization completed. ===


In [23]:
# Evaluation on test set - AFTER quantization
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids']
        labels = batch['label']
        
        outputs = model(input_ids)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
accuracy = correct / total
print(f"Quantization Test Accuracy: {accuracy:.4f}")

Quantization Test Accuracy: 0.7886


In [24]:
sample_text = """Pham Minh Chinh is the prime minister of the Vietnam goverment."""
# sample_text = """The new travel guidelines have been released for tourists. Vietnam is welcoming visitors back."""
# sample_text = """Pho is a popular dish, which is a noodle soup consisting of broth, rice noodles, herbs, and meat."""
sample_encoding = tokenizer(
    sample_text,
    max_length=max_length,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

sample_input_ids = sample_encoding['input_ids']
model.eval()
with torch.no_grad():
    sample_output = model(sample_input_ids)
    _, sample_predicted = torch.max(sample_output.data, 1)
    predicted_category = le.inverse_transform(sample_predicted.numpy())[0]
    
print(f"Sample Text: {sample_text}")
print(f"Predicted Category: {predicted_category}")

Sample Text: Pham Minh Chinh is the prime minister of the Vietnam goverment.
Predicted Category: POLITICS


In [25]:
# Save model except embedding layer
state_dict = model.state_dict()

# Remove the embedding weights
state_dict_no_embed = {k: v for k, v in state_dict.items() if not k.startswith("embedding.")}

# Save
output_path = os.path.join("models", "news_classifier_quantized_no_embedding.pth")
torch.save(state_dict_no_embed, output_path)

print(f"Model size without embedding layer. File size: {os.path.getsize(output_path) / (1024 * 1024):.2f} MB") 

Model size without embedding layer. File size: 0.12 MB
